In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import sys, os

root_dir = "../../"
sys.path.insert(0, root_dir)

from spec.enums import MainTableColumns as Cols
from datasets.cs1eng.configs import S24

In [2]:
from database.config import PS2DataConfig
from spec.spec_definition import PS2Versions


config = S24

data_config_path = os.path.join(root_dir, "../sample_data_configs", f"{config.name}.yaml")
spec = PS2Versions.v1_0.load()
data_config = PS2DataConfig.from_yaml(data_config_path, spec)

data_config


PS2DataConfig(root_path='D:/GitHub Large/AIF-Analysis/data/CSC111/s24/', metadata=MetadataValues(Version='1.0', IsEventOrderingConsistent=True, EventOrderScope='Global', EventOrderScopeColumns='', CodeStateRepresentation='Table', ProgramInputLinkTable=''), optimize_codestate_ids=True, codestates_have_sections=True, main_table_file=None, codestates_table_relative_path='CodeStates.csv', metadata_file='DatasetMetadata.csv', sqlalchemy_url='sqlite:///D:/GitHub Large/AIF-Analysis/data/CSC111/s24/Logging.db', short_str_length=255, path_str_length=2048, echo=False)

In [3]:
from analytics.ps2_dataset import PS2Dataset

dataset = PS2Dataset(spec, data_config)

In [ ]:
hw3 = pd.read_csv("hw3.csv")

In [5]:
# main_table = dataset.get_main_table()
# main_table.head()

In [6]:
# hw3 = main_table[main_table[Cols.ProblemID] == "homework3"]
# hw3.shape
# hw3.to_csv("hw3.csv", index=False)

In [ ]:
csids = list(range(10000))
codestates = dataset.get_codestates(csids)
codestates.head()

,CodeStateID,Code,CodeStateID
0,1,"""""""\r\n Author: [First_Name Last_Name]\r\n ...",1
1,2,"""""""\r\n Author: [First_Name Last_Name]\r\n ...",2
2,3,"""""""\r\n Author: [First_Name Last_Name]\r\n ...",3
3,4,"""""""\r\n Author: [First_Name Last_Name]\r\n ...",4
4,5,"\r\n\r\nclass_name = input(""What is the name o...",5


In [44]:
all_code = codestates.Code.tolist()

In [45]:
import re

occurrences = {}
for code in all_code:
    # Split by whitespace, punctuation, underscores, dashes, etc.
    words = re.findall(r'\b\w+\b', code)
    for word in words:
        word = word.lower()
        sub_words = word.split("_")
        for sub_word in sub_words:
            if sub_word not in occurrences:
                occurrences[sub_word] = 0
            occurrences[sub_word] += 1


In [46]:
len(occurrences)

33407

In [ ]:
class TrieNode:
    def __init__(self, parent, ch):
        self.children = {}
        self.is_end = False
        self.parent = parent
        self.character = ch

    def get_word(self):
        # Get the word represented by this node
        chars = []
        node = self
        while node is not None:
            chars.append(node.character)
            node = node.parent
        return ''.join(reversed(chars))

class SupersetTrie:
    def __init__(self):
        self.root = TrieNode(None, '')

    def insert(self, word: str):
        node = self.root
        for ch in word:
            if ch not in node.children:
                node.children[ch] = TrieNode(node, ch)
            node = node.children[ch]
        node.is_end = True

    def has_superset(self, word: str) -> bool:
        node = self.root
        for ch in word:
            if ch not in node.children:
                return False
            node = node.children[ch]
        desc = self._get_first_descendant(node)
        if desc is not None:
            # print(f"Found superset of {word}: {desc.get_word()}")
            return True

    def _get_first_descendant(self, node: TrieNode) -> bool:
        # Is there any complete word that continues from here?
        if node is None:
            return False
        for child in node.children.values():
            if child.is_end or self._get_first_descendant(child):
                return child
        return None

def remove_prefix_words(word_freq_dict):
    trie = SupersetTrie()
    for word in word_freq_dict:
        trie.insert(word)

    filtered = {
        word: freq for word, freq in word_freq_dict.items()
        if not trie.has_superset(word)
    }
    return filtered

In [ ]:
# Only seems to halve the number of words
filtered = remove_prefix_words(occurrences)
len(filtered)

18426